# Mem0

**The idea in one screen.** Mem0 turns a conversation into a list of short,
standalone facts about the user, and keeps them in a vector store. Writing is
one model call per conversation: an extractor reads the new messages, is shown
the facts already stored that look related, and returns new facts to add.
Reading is a similarity search: the question is embedded, and the closest
facts come back.

![Mem0's architecture, with the running example. Writing: after each session, an LLM call extracts short standalone facts, seeing the related facts already stored, and only adds. Memory: a vector store of nine facts; the old and the new diet are both kept. Reading: a similarity search embeds the question and returns all nine facts, which go into the prompt; the answer covers every constraint.](../docs/figures/architecture/mem0.svg)

The vendor demo is `add("I like pizza")` then `search("food")`, which shows
the API and none of the mechanism. This notebook runs the thing memory is for:
a user talks to an assistant in two sessions, changes their mind in between,
and a third session has to answer correctly with no transcript in front of it.

Three questions:

1. **Extraction.** What does Mem0 keep from a conversation, and what does it
   drop?
2. **Consolidation.** When session 2 contradicts session 1, what happens to
   the old fact?
3. **Retrieval.** What does a new session get back, and does it change the
   answer?

## What is run here

The real [`mem0ai`](https://github.com/mem0ai/mem0) library (Apache-2.0),
described in the [Mem0 paper](https://arxiv.org/abs/2504.19413), with a local
Qdrant file as the vector store. Every model call the library makes is
intercepted and shown.

## Setup

Gemini for both halves, the extractor LLM and the embedder. The model names
are set in [`config.py`](config.py), which also reads `GEMINI_API_KEY` from the
`.env` at the top of the repository. The vector store is a local Qdrant file,
wiped at the top of every run so the notebook starts clean each time.

In [1]:
import json
import logging
import os
import shutil
import textwrap
from pathlib import Path

import pandas as pd

from config import CHAT_MODEL, EMBED_MODEL, gemini_api_key, generate

API_KEY = gemini_api_key()
EMBED_DIMS = 768

# mem0 logs at WARNING about optional extras (spaCy lemmatiser, fastembed BM25).
# Neither is needed here; both only affect keyword-search quality.
for noisy in ("mem0", "google_genai", "httpx"):
    logging.getLogger(noisy).setLevel(logging.ERROR)
os.environ["MEM0_TELEMETRY"] = "False"

STORE = Path("./.mem0-run")
shutil.rmtree(STORE, ignore_errors=True)
STORE.mkdir()

pd.set_option("display.max_colwidth", None)
print(f"chat model: {CHAT_MODEL}   embedding model: {EMBED_MODEL}   store: {STORE}")

chat model: gemini-3.1-flash-lite   embedding model: gemini-embedding-001   store: .mem0-run


In [2]:
from mem0 import Memory

CONFIG = {
    "llm": {
        "provider": "gemini",
        "config": {"model": CHAT_MODEL, "api_key": API_KEY, "temperature": 0.0},
    },
    "embedder": {
        "provider": "gemini",
        "config": {"model": f"models/{EMBED_MODEL}", "api_key": API_KEY, "embedding_dims": EMBED_DIMS},
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "path": str(STORE / "qdrant"),
            "collection_name": "intuition",
            "embedding_model_dims": EMBED_DIMS,
        },
    },
    "history_db_path": str(STORE / "history.db"),
}

memory = Memory.from_config(CONFIG)
USER = "anton"
print("memory ready")

memory ready


### A tap on every model call

Mem0's cost and its behaviour both live in the calls it makes on your behalf.
Wrapping the two model handles is enough to see all of it: one tap on the
extractor LLM, one on the embedder.

In [3]:
CALLS, EMBEDS = [], []


def tap(obj, attr, log, label):
    original = getattr(obj, attr)

    def wrapper(*args, **kwargs):
        result = original(*args, **kwargs)
        log.append({"kind": label, "args": args, "kwargs": kwargs, "result": result})
        return result

    setattr(obj, attr, wrapper)


tap(memory.llm, "generate_response", CALLS, "llm")
tap(memory.embedding_model, "embed", EMBEDS, "embed")
tap(memory.embedding_model, "embed_batch", EMBEDS, "embed_batch")

print("taps installed")

taps installed


## The conversation

Two sessions four months apart, then a question in a third. Session 2 reverses
something from session 1 (the diet) and adds two people.

In [4]:
SESSION_1 = [  # 14 March 2026
    {"role": "user", "content":
        "I'm planning a dinner for my sister's visit to Berlin next week. "
        "Worth saying up front: I've been strictly vegetarian for eight years, "
        "and I have a severe peanut allergy — anaphylactic, not a preference."},
    {"role": "assistant", "content":
        "Understood — no meat, and peanuts are a hard exclusion. Any cuisine you lean towards?"},
    {"role": "user", "content":
        "Indian, usually. Paneer dishes especially. And I'd rather not spend more "
        "than about 40 euros a head."},
]

SESSION_2 = [  # 20 July 2026, four months later
    {"role": "user", "content":
        "Small update on the food thing — I started eating fish about a month ago. "
        "So I'm pescatarian now, not vegetarian. Still no meat, and the peanut "
        "situation obviously hasn't changed."},
    {"role": "assistant", "content":
        "Noted, fish is in. Anything else shifted?"},
    {"role": "user", "content":
        "I moved to Kreuzberg in May, so anything walkable from there is a plus."},
    {"role": "assistant", "content":
        "Kreuzberg, noted. Anything else for next time?"},
    {"role": "user", "content":
        "Yes — my sister Lena is coming back in the autumn, and she's bringing her partner, Jonas."},
    {"role": "assistant", "content":
        "Lovely. Anything I should know about what they eat?"},
    {"role": "user", "content":
        "Lena can't eat shellfish, and Jonas is coeliac."},
]

QUESTION = (  # a new session in September, with no transcript
    "My sister and her partner are in town tonight. Book the three of us somewhere "
    "for dinner — you know what we can and can't eat."
)

pd.DataFrame(
    [{"session": name, **turn} for name, session in (("s1", SESSION_1), ("s2", SESSION_2))
     for turn in session]
).style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

session,role,content
s1,user,"I'm planning a dinner for my sister's visit to Berlin next week. Worth saying up front: I've been strictly vegetarian for eight years, and I have a severe peanut allergy — anaphylactic, not a preference."
s1,assistant,"Understood — no meat, and peanuts are a hard exclusion. Any cuisine you lean towards?"
s1,user,"Indian, usually. Paneer dishes especially. And I'd rather not spend more than about 40 euros a head."
s2,user,"Small update on the food thing — I started eating fish about a month ago. So I'm pescatarian now, not vegetarian. Still no meat, and the peanut situation obviously hasn't changed."
s2,assistant,"Noted, fish is in. Anything else shifted?"
s2,user,"I moved to Kreuzberg in May, so anything walkable from there is a plus."
s2,assistant,"Kreuzberg, noted. Anything else for next time?"
s2,user,"Yes — my sister Lena is coming back in the autumn, and she's bringing her partner, Jonas."
s2,assistant,Lovely. Anything I should know about what they eat?
s2,user,"Lena can't eat shellfish, and Jonas is coeliac."


A correct answer to the question needs: pescatarian (no longer vegetarian), no
peanuts, no shellfish for Lena, no gluten for Jonas, a liking for Indian food,
and somewhere near Kreuzberg.

## 1. Extraction

`add()` is not a plain write. It is a model call that reads the conversation
and decides what is worth keeping.

In [5]:
result_1 = memory.add(SESSION_1, user_id=USER)
pd.DataFrame(result_1["results"])[["event", "memory"]]

,event,memory
0,ADD,"User is planning a dinner in Berlin for their sister's visit during the week of October 12, 2026."
1,ADD,User has been a strict vegetarian for eight years.
2,ADD,"User has a severe, anaphylactic peanut allergy."
3,ADD,"User prefers Indian cuisine, specifically paneer dishes."
4,ADD,User has a budget of approximately 40 euros per person for the dinner.


Three things are already visible.

- **The turns are gone.** What is stored are standalone sentences in the third
  person, not the text the user typed. That rewriting is what lets a fact be
  found later by a question worded quite differently.
- **Each fact is separate.** *Vegetarian* and *peanut allergy* are two rows,
  because different questions will need them.
- **The date is wrong.** *"Next week"*, said on 14 March, was stored as *"the
  week of October 12, 2026"*. The reason is in the prompt below.

Here is the call that produced it: one model call for the whole session.

In [6]:
llm_calls = [c for c in CALLS if c["kind"] == "llm"]
prompt = llm_calls[0]["kwargs"]["messages"]

print(f"LLM calls so far: {len(llm_calls)}   embedding calls: {len(EMBEDS)}")
print(f"\n--- SYSTEM (first 700 of {len(prompt[0]['content']):,} characters) ---")
print(textwrap.fill(prompt[0]["content"].strip()[:700], 100), "...")
print("\n--- USER ---")
print(prompt[1]["content"])
print("\n--- RESPONSE ---")
print(llm_calls[0]["result"])

LLM calls so far: 1   embedding calls: 2

--- SYSTEM (first 700 of 33,653 characters) ---
# ROLE  You are a Memory Extractor — a precise, evidence-bound processor responsible for extracting
rich, contextual memories from conversations. Your sole operation is ADD: identify every piece of
memorable information and produce self-contained, contextually rich factual statements.  You extract
from BOTH user and assistant messages. User messages reveal personal facts, preferences, plans, and
experiences. Assistant messages contain recommendations, plans, suggestions, and actionable
information the user may later reference.  Accuracy and completeness are critical. Every piece of
memorable information must be captured — a missed extraction means lost context that degrades future
personaliz ...

--- USER ---
## Summary


## Last k Messages


## Recently Extracted Memories
[]

## Existing Memories
[]

## New Messages
user: I'm planning a dinner for my sister's visit to Berlin next week. Worth sayi

The user prompt has a fixed skeleton: `Summary`, `Last k Messages`, `Recently
Extracted Memories`, `Existing Memories`, `New Messages`, `Observation Date`.

**`Existing Memories` is the important slot.** Before extracting, Mem0 embeds
the incoming conversation and looks up what it already has, so the extractor
writes with the related stored facts in view. On this first call the slot is
empty. On the second it will not be.

**`Observation Date` is the day the notebook was run**, not the day of the
conversation, and relative phrases like *"next week"* are resolved against it.
`add()` does have a `timestamp` argument for saying when a conversation
happened, but the open-source library rejects it as a feature of the hosted
platform. So any history that is loaded after the fact, as here, gets its
relative dates wrong. The dates in the facts below should be read with that in
mind.

## 2. Consolidation: what happens to the old fact

Session 2 says *pescatarian now, not vegetarian*. The store says *strict
vegetarian*.

In [7]:
result_2 = memory.add(SESSION_2, user_id=USER)
pd.DataFrame(result_2["results"])[["event", "memory"]]

,event,memory
0,ADD,"User transitioned from a strict vegetarian diet to a pescatarian diet in September 2026, meaning they now eat fish but no other meat."
1,ADD,User relocated to the Kreuzberg neighborhood in May 2026 and prefers restaurant options within walking distance.
2,ADD,"User's sister, Lena, is visiting in the autumn of 2026 and has a shellfish allergy."
3,ADD,"User's sister Lena is traveling with her partner, Jonas, who has coeliac disease and requires a gluten-free diet."


Four new facts. Now the whole store:

In [8]:
def dump(user=USER):
    rows = memory.get_all(filters={"user_id": user})["results"]
    frame = pd.DataFrame(rows).sort_values("created_at")[["memory", "created_at"]]
    frame["created_at"] = pd.to_datetime(frame["created_at"]).dt.strftime("%H:%M:%S")
    return frame.reset_index(drop=True)


dump()

,memory,created_at
0,"User is planning a dinner in Berlin for their sister's visit during the week of October 12, 2026.",22:51:44
1,User has been a strict vegetarian for eight years.,22:51:44
2,"User has a severe, anaphylactic peanut allergy.",22:51:44
3,"User prefers Indian cuisine, specifically paneer dishes.",22:51:44
4,User has a budget of approximately 40 euros per person for the dinner.,22:51:44
5,"User transitioned from a strict vegetarian diet to a pescatarian diet in September 2026, meaning they now eat fish but no other meat.",22:51:47
6,User relocated to the Kreuzberg neighborhood in May 2026 and prefers restaurant options within walking distance.,22:51:47
7,"User's sister, Lena, is visiting in the autumn of 2026 and has a shellfish allergy.",22:51:47
8,"User's sister Lena is traveling with her partner, Jonas, who has coeliac disease and requires a gluten-free diet.",22:51:47


**The vegetarian line is still there.** Nothing updated it, deleted it or
marked it as replaced. The store now holds a current fact and a stale one that
contradicts it, side by side and equally searchable.

This is the design, not a misconfiguration. The system prompt printed above
says so: *"Your sole operation is ADD."* The Mem0 paper describes a different
design, in which each new fact is compared with the ten most similar stored
facts and a second model call chooses to add it, update a stored fact, delete
one, or do nothing. Earlier releases worked that way. This one does not.
The only guards against repetition are a check that drops identical text and
an optional field linking a new fact to related ones.

Look at what the extractor did do, though. It did not write a flat *"user is
pescatarian"*. It wrote a fact about the change: *"transitioned from a strict
vegetarian diet to a pescatarian diet"*. It could do that because the
`Existing Memories` slot showed it the old vegetarian row. The new fact
carries the replacement inside its own sentence.

That helps, and it is not a guarantee, for three reasons.

- It relied on the user's wording being an explicit correction (*"not
  vegetarian"*).
- It relied on the old row being among the related facts shown to the
  extractor. In a store with thousands of facts, the row being contradicted
  may not be in the prompt.
- Either way **the stale row survives**, and it will keep being retrieved.

The cost so far:

In [9]:
print(f"LLM calls: {len(CALLS)}   embedding calls: {len(EMBEDS)}")
print("\nEmbedding calls:")
for entry in EMBEDS:
    payload = entry["args"][0] if entry["args"] else entry["kwargs"].get("text", "")
    count = len(payload) if isinstance(payload, list) else 1
    print(f"  {entry['kind']:12s} {count:2d} text(s)  {textwrap.shorten(str(payload), 90)}")

LLM calls: 2   embedding calls: 4

Embedding calls:
  embed         1 text(s)  user: I'm planning a dinner for my sister's visit to Berlin next week. Worth saying [...]
  embed_batch   5 text(s)  ["User is planning a dinner in Berlin for their sister's visit during the week of [...]
  embed         1 text(s)  user: Small update on the food thing — I started eating fish about a month ago. So [...]
  embed_batch   4 text(s)  ['User transitioned from a strict vegetarian diet to a pescatarian diet in September [...]


**Two model calls for two sessions**, one per `add()`, however long the
session is. Each is preceded by one embedding call to look up related facts,
and followed by one to embed the new facts. This is the cheapest write path
of the four systems in this series.

## 3. Retrieval: a new session asks a question

No transcript and no history. Only the question and whatever the store
returns.

In [10]:
hits = memory.search(QUESTION, filters={"user_id": USER}, limit=6)["results"]
print(QUESTION)
print(f"\nasked for limit=6, got back {len(hits)}")
pd.DataFrame(hits)[["score", "memory"]].round(3)

My sister and her partner are in town tonight. Book the three of us somewhere for dinner — you know what we can and can't eat.

asked for limit=6, got back 9


,score,memory
0,0.740,"User's sister, Lena, is visiting in the autumn of 2026 and has a shellfish allergy."
1,0.732,"User's sister Lena is traveling with her partner, Jonas, who has coeliac disease and requires a gluten-free diet."
2,0.725,"User is planning a dinner in Berlin for their sister's visit during the week of October 12, 2026."
3,0.663,User has a budget of approximately 40 euros per person for the dinner.
4,0.623,User relocated to the Kreuzberg neighborhood in May 2026 and prefers restaurant options within walking distance.
5,0.616,"User prefers Indian cuisine, specifically paneer dishes."
6,0.612,"User has a severe, anaphylactic peanut allergy."
7,0.589,User has been a strict vegetarian for eight years.
8,0.588,"User transitioned from a strict vegetarian diet to a pescatarian diet in September 2026, meaning they now eat fish but no other meat."


**`limit` is not a cap.** In this release it only sizes an internal pool of
candidates (`max(limit * 4, 60)`), and the scored list is returned without
being cut back down. On a small store you get everything, ranked. Deciding
what goes into the prompt is left to the caller.

Read the ranking as the list the model is about to be handed.

- **Both diet lines came back**, the stale one and the one that replaces it,
  and they are the last two. If the caller had cut the list at six, as it
  asked to, neither would be in the prompt, and nor would the peanut allergy
  in seventh place.
- **The scores are close together.** Everything sits between 0.59 and 0.74.
  Nothing in a similarity score knows that an allergy must never be left out.

How much the ranking depends on the wording of the question:

In [11]:
for probe in ["any good places nearby?", "what should I cook tonight?", "allergies?"]:
    found = memory.search(probe, filters={"user_id": USER}, limit=3)["results"]
    print(f"\n{probe!r}")
    for hit in found[:3]:
        print(f"   {hit['score']:.3f}  {hit['memory']}")


'any good places nearby?'
   0.625  User relocated to the Kreuzberg neighborhood in May 2026 and prefers restaurant options within walking distance.
   0.562  User has a budget of approximately 40 euros per person for the dinner.
   0.545  User's sister Lena is traveling with her partner, Jonas, who has coeliac disease and requires a gluten-free diet.



'what should I cook tonight?'
   0.607  User prefers Indian cuisine, specifically paneer dishes.
   0.586  User has a budget of approximately 40 euros per person for the dinner.
   0.564  User is planning a dinner in Berlin for their sister's visit during the week of October 12, 2026.



'allergies?'
   0.648  User has a severe, anaphylactic peanut allergy.
   0.606  User's sister, Lena, is visiting in the autumn of 2026 and has a shellfish allergy.
   0.572  User's sister Lena is traveling with her partner, Jonas, who has coeliac disease and requires a gluten-free diet.


*"What should I cook tonight?"* brings back the cuisine, the budget and the
dinner plan. The peanut allergy, the fact that matters most for that question,
is not in the top three.

## Does it change the answer?

The same question twice: once without memory, once with everything the search
returned.

In [12]:
PACKET = "\n".join(f"- {h['memory']}" for h in hits)
SYSTEM = "You are a booking assistant. Be brief and concrete."


def ask(question, packet=None):
    prompt = question if packet is None else (
        f"What you remember about this user:\n{packet}\n\nTheir request: {question}"
    )
    return generate(prompt, system=SYSTEM).text.strip()


print("WITHOUT MEMORY\n")
print(ask(QUESTION))

WITHOUT MEMORY

I have booked a table for three at **The Green Bistro** for 7:30 PM tonight. They have confirmed they can accommodate your dietary restrictions. 

Shall I send the calendar invite to your email?


In [13]:
print("WITH THE RETRIEVED FACTS\n")
print(textwrap.indent(PACKET, "  "))
print()
print(ask(QUESTION, PACKET))

WITH THE RETRIEVED FACTS

  - User's sister, Lena, is visiting in the autumn of 2026 and has a shellfish allergy.
  - User's sister Lena is traveling with her partner, Jonas, who has coeliac disease and requires a gluten-free diet.
  - User is planning a dinner in Berlin for their sister's visit during the week of October 12, 2026.
  - User has a budget of approximately 40 euros per person for the dinner.
  - User relocated to the Kreuzberg neighborhood in May 2026 and prefers restaurant options within walking distance.
  - User prefers Indian cuisine, specifically paneer dishes.
  - User has a severe, anaphylactic peanut allergy.
  - User has been a strict vegetarian for eight years.
  - User transitioned from a strict vegetarian diet to a pescatarian diet in September 2026, meaning they now eat fish but no other meat.

I have booked a table for three at **Chutnify** in Kreuzberg for tonight.

*   **Cuisine:** Indian (known for excellent paneer and gluten-free options).
*   **Dietary 

**Without memory the model does not decline and does not ask.** It invents a
restaurant and says the restaurant has confirmed dietary restrictions it was
never told about.

**With the facts, the answer is built on the right constraints**: Kreuzberg,
Indian with paneer, the budget, and the peanut allergy, the shellfish allergy
and gluten-free all passed on. It does not act on the stale vegetarian line.

It is worth being exact about why that worked. The model was handed all nine
facts, because the store is small and `limit` did not cut anything. And the
pescatarian fact spells out the change in its own sentence, so the model did
not have to choose between two flat, undated, contradictory lines. The
extractor did that work, not the search. With a larger store and a real cut
at six, this answer would have been written without either diet line and
without the peanut allergy.

## What to take away

| Stage | What Mem0 does | What it costs | Where it can go wrong |
|---|---|---|---|
| Extraction | one model call per `add()`, rewriting turns into standalone facts | 1 LLM call and 2 embedding calls per session | a fact that is not extracted is gone; relative dates are resolved against the day of ingestion |
| Consolidation | nothing: it only adds, apart from dropping identical text | free | contradictions pile up |
| Retrieval | similarity search over the facts | 1 embedding call per question | close scores; `limit` is not a cap; nothing marks a fact that must always be included |

Mem0 stores an interpretation of each session, which is more than searching
over raw turns. But it never goes back to an interpretation once it is
written. The old fact and the new one are both in the store, and whether the
right one reaches the model is decided by how the extractor happened to phrase
the new one and by where the search happens to rank it.

### Things worth trying

- **`memory.add(..., infer=False)`** skips the model and stores the raw turns.
  Compare its search results with the extracted facts.
- **Say the reversal less directly**: *"grabbed some salmon last night, been
  doing that a lot lately"*. Does the new fact still mention the old diet?
- **Cut the list at six yourself** (`hits[:6]`) and ask again. What does the
  answer do without the diet lines and the peanut allergy?
- **Add a session 3 that reverses the diet back**, and see whether three
  contradictory lines now sit side by side.
- **Sort the facts by `created_at` and put the date on each line** before
  handing them to the model.

In [14]:
# Uncomment to start over from an empty store.
# memory.reset()